# 1

In [73]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructField, StructType, StringType, StringType, IntegerType

spark = SparkSession.builder.appName("trst").getOrCreate()

schema = StructType([
    StructField("id", IntegerType()),
    StructField('name', StringType()),
    StructField("age", IntegerType()),
    StructField('city', StringType())

])

df = spark.read.option("sep", "\t").csv("1.csv", header=True, schema=schema)

df.createOrReplaceTempView("temp")


df = df.drop_duplicates()
df.printSchema()

# getting the average age
avg_age = df.select(func.avg(func.col("age"))).first()[0]

# filling in null ages with avg_age
df = df.fillna({'age': int(avg_age)})
df = df.fillna({'name': "Unknown"})

result = df.select(func.col('name'), func.col('age'), func.col('city')).show()

26/07/10 12:43:35 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)

+-----+---+----+
| name|age|city|
+-----+---+----+
|Alice| 25|  NY|
| null| 40|  NY|
|  Bob| 32|  LA|
+-----+---+----+



# 2

In [74]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructField, StructType, StringType, StringType, IntegerType

spark = SparkSession.builder.appName("trst").getOrCreate()

df = spark.read.option("sep", ',').csv("2.csv", header=True, inferSchema=True)

df.printSchema()

print("total spent per customer")
grouped = df.groupBy(func.col("customer")).agg(func.sum(func.col("amount")).alias("total_spent"))
df.show()

print('highest spender')
highest_spender = grouped.sort(func.col('total_spent'), ascending=False).limit(1)
highest_spender.show()

print("average purchase amount")
average = df.agg(func.avg('amount')).first()[0]
print(f"{average:.2f}")




root
 |-- customer: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)

total spent per customer
+--------+----------+------+
|customer|   product|amount|
+--------+----------+------+
|       A|     Phone|   500|
|       A|    Laptop|  1000|
|       A|    Tablet|   450|
|       B|     Phone|   700|
|       B|Headphones|   150|
|       B|    Laptop|  1200|
|       C|     Phone|   650|
|       C|   Monitor|   300|
|       C|  Keyboard|    80|
|       D|    Laptop|  1100|
|       D|     Mouse|    40|
|       D|     Phone|   550|
|       E|    Tablet|   500|
|       E|   Monitor|   350|
|       E|Headphones|   200|
|       F|     Phone|   750|
|       F|    Laptop|  1300|
|       F|  Keyboard|    90|
|       G|   Monitor|   400|
|       G|     Mouse|    35|
+--------+----------+------+
only showing top 20 rows
highest spender
+--------+-----------+
|customer|total_spent|
+--------+-----------+
|       K|       2190|
+--------+-----------+

# 3

In [75]:
from pyspark.sql import SparkSession, functions as f 
from pyspark.sql.window import Window


spark = SparkSession.builder.appName("tesst").getOrCreate()

df = spark.read.option("sep", ",").csv("3.csv", header=True, inferSchema=True)

df.printSchema()

# make the window function
window_spec = Window.partitionBy("department").orderBy(f.col("salary").desc())

# rank the employees
result = (
    df.withColumn("rank", func.row_number().over(window_spec))
    .filter(func.col("rank") == 1)
)

print("highees paied employees per department")
result.drop("rank").show()


root
 |-- department: string (nullable = true)
 |-- employee: string (nullable = true)
 |-- salary: integer (nullable = true)

highees paied employees per department
+-----------+--------+------+
| department|employee|salary|
+-----------+--------+------+
|Engineering|    Liam|   420|
|    Finance|  Robert|   350|
|         HR|   Chris|   190|
|         IT|   Sarah|   250|
|  Marketing|  Olivia|   240|
| Operations|   Ethan|   260|
|      Sales|     Mia|   310|
|    Support|   Logan|   190|
+-----------+--------+------+



26/07/10 12:43:36 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


# 4

In [76]:
from pyspark.sql import SparkSession, functions as f 
from pyspark.sql.types import StructType, StructField

spark = SparkSession.builder.appName("test").getOrCreate()

df_customers = spark.read.option("sep", ",").csv("4_customers.csv", header=True, inferSchema=True)
df_orders = spark.read.csv("4_orders.csv", header=True, inferSchema=True)

df_customers.printSchema()
df_orders.printSchema()

# do an inner join on the 2 tables
join = df_customers.join(df_orders, df_customers.id == df_orders.customer_id, "inner").show()

# get customers with no orders
left_join = df_customers.join(df_orders, df_customers.id == df_orders.customer_id, "left")

print("customers without orders")
left_join.filter(func.col('amount').isNull()).show()


total = left_join.groupBy(func.col("id"), func.col("name")).agg(func.sum(func.col("amount")).alias("total_spent"))

total.drop("id").orderBy(func.col("total_spent").desc()).show()

26/07/10 12:43:36 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- amount: integer (nullable = true)

+---+-------+-----------+------+
| id|   name|customer_id|amount|
+---+-------+-----------+------+
|  1|  Alice|          1|   100|
|  1|  Alice|          1|    50|
|  2|    Bob|          2|    75|
|  2|    Bob|          2|   125|
|  3|Charlie|          3|    25|
|  3|Charlie|          3|   200|
|  4|  Diana|          4|   300|
|  4|  Diana|          4|   150|
|  5|    Eve|          5|    80|
|  5|    Eve|          5|    60|
|  6|  Frank|          6|   500|
|  6|  Frank|          6|   120|
|  9|    Ivy|          9|    70|
|  9|    Ivy|          9|    95|
+---+-------+-----------+------+

customers without orders
+---+-----+-----------+------+
| id| name|customer_id|amount|
+---+-----+-----------+------+
|  7|Grace|       NULL|  NULL|
|  8|Henry|       NULL|  NULL|
| 10| Jack|       NULL|  NULL|
+---+-----+-----------+-----

# 5

In [77]:
from pyspark.sql import SparkSession, functions as f 
from pyspark.sql.types import StructField, StructType


spark = SparkSession.builder.appName("test").getOrCreate()

schema = StructType([
    StructField("email", StringType())
])

df = spark.read.csv("5.csv", header=True, schema=schema)

df.groupBy("email").agg(func.count("email").alias("times_shown")).sort("times_shown", ascending=False).show()

+----------------+-----------+
|           email|times_shown|
+----------------+-----------+
|  alice@test.com|          3|
|    bob@test.com|          3|
|   emma@test.com|          2|
|   jack@test.com|          2|
|charlie@test.com|          2|
|  maria@test.com|          2|
| rachel@test.com|          1|
|  nancy@test.com|          1|
| oliver@test.com|          1|
|  grace@test.com|          1|
|  wendy@test.com|          1|
| victor@test.com|          1|
|    tom@test.com|          1|
|  david@test.com|          1|
|   luke@test.com|          1|
|  quinn@test.com|          1|
|  henry@test.com|          1|
|   paul@test.com|          1|
|  sarah@test.com|          1|
|   kate@test.com|          1|
+----------------+-----------+
only showing top 20 rows


# 6

In [78]:
from pyspark.sql import SparkSession, functions as f 
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
df = spark.read.csv("6.csv", header=True, inferSchema=True)

# unbounded proceeding - very first row in the partition winodow
# rowsBetween - makes the calculation only on the rows between the spcified rows
window_spec = (Window.partitionBy("user").orderBy("date").
               rowsBetween(Window.unboundedPreceding, Window.currentRow)
            )

# getting the previouse row according to date
window_spec_prev = (Window.partitionBy("user").orderBy("date"))


# makes a new collumn called running_total and performs a sum on all the partitioned rows
result = (df.withColumn("running_total", f.sum("amount").over(window_spec))).show()

# gets the previous transaction using lag (put what you want in the prev and how far back)
prev = df.withColumn("previous_transaction", f.lag("amount", 1).over(window_spec_prev))

prev.show()

# using coalesce to default to a 0 value
# need to use lit(0) pyspark can not do operations on regular python vars (lit(0) creates a column with 0 being the value)
difference_prev = prev.withColumn("difference", f.coalesce(
        f.abs(f.col("amount") - f.col("previous_transaction")),
        f.lit(0)
    ))

difference_prev.show()

+----+-----+------+-------------+
|user| date|amount|running_total|
+----+-----+------+-------------+
|   A| Jan1|    10|           10|
|   A|Jan10|    60|           70|
|   A| Jan2|    20|           90|
|   A| Jan3|    15|          105|
|   A| Jan4|    30|          135|
|   A| Jan5|    25|          160|
|   A| Jan6|    40|          200|
|   A| Jan7|    35|          235|
|   A| Jan8|    50|          285|
|   A| Jan9|    45|          330|
|   B| Jan1|    15|           15|
|   B|Jan10|    70|           85|
|   B| Jan2|    25|          110|
|   B| Jan3|    10|          120|
|   B| Jan4|    35|          155|
|   B| Jan5|    50|          205|
|   B| Jan6|    45|          250|
|   B| Jan7|    30|          280|
|   B| Jan8|    55|          335|
|   B| Jan9|    65|          400|
+----+-----+------+-------------+
only showing top 20 rows
+----+-----+------+--------------------+
|user| date|amount|previous_transaction|
+----+-----+------+--------------------+
|   A| Jan1|    10|                N

# 7

In [79]:
from pyspark.sql import SparkSession, functions as f 
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
df = spark.read.csv("7.csv", header=True, inferSchema=True)


# unbounded proceeding - very first row in the partition winodow
# rowsBetween - makes the calculation only on the rows between the spcified rows
# paritions by user and gets from the first column to the current one
window_spec = (Window.partitionBy("user").orderBy(f.to_date("timestamp")).
               rowsBetween(Window.unboundedPreceding, Window.currentRow)
            )

# ordering the rows by data per user
window_spec_prev = (Window.partitionBy("user").orderBy(f.to_date("timestamp")))




# makes a new collumn called running_total and performs a sum on all the partitioned rows
result = (df.withColumn("running_total", f.sum("amount").over(window_spec))).show()

# gets the previous transaction using lag (put what you want in the prev and how far back)
prev = df.withColumn("previous_transaction", f.lag("amount", 1).over(window_spec_prev))

prev.show()

# using coalesce to default to a 0 value
# need to use lit(0) pyspark can not do operations on regular python vars (lit(0) creates a column with 0 being the value)
difference_prev = prev.withColumn("difference", f.coalesce(
        f.abs(f.col("amount") - f.col("previous_transaction")),
        f.lit(0)
    ))

# gets the previous timestamp
difference_prev = difference_prev.withColumn("previous_timestamp", f.lag("timestamp").over(window_spec_prev))

# gets the difference between the previous timestamp and the current one
difference_prev = difference_prev.withColumn("minutes_since_last", f.coalesce(f.floor(
    (
        (f.col("timestamp").cast("long")) - 
        (f.col("previous_timestamp").cast("long"))) / 60
    ), 
        f.lit(0)))

# if there is a new session then return 1, otherwise return 0
difference_prev = difference_prev.withColumn("new_session", f.when(
    (f.col("minutes_since_last")>30.0) | 
    (f.col("minutes_since_last") == 0), 1).otherwise(0) 
    )

# add up all the session per user between the current time and the previous times to see what session # they are on
find_running_session = difference_prev.withColumn("session_number", f.sum("new_session").over(window_spec))

# drop the unnecessary columns that we dont need anymore
find_running_session.drop("previous_timestamp", "new_session").show()

+----+-------------------+------+-------------+
|user|          timestamp|amount|running_total|
+----+-------------------+------+-------------+
|   A|2026-01-01 08:15:23|    10|           10|
|   A|2026-01-01 08:42:11|    25|           35|
|   A|2026-01-02 09:30:45|    15|           50|
|   A|2026-01-02 09:55:09|    30|           80|
|   A|2026-01-03 18:55:31|    12|           92|
|   A|2026-01-03 19:12:17|    22|          114|
|   A|2026-01-05 16:45:02|    18|          132|
|   A|2026-01-05 17:30:02|    40|          172|
|   A|2026-01-07 11:34:10|    16|          188|
|   A|2026-01-07 11:50:28|    28|          216|
|   B|2026-01-01 09:10:05|     8|            8|
|   B|2026-01-01 09:25:44|    20|           28|
|   B|2026-01-02 13:25:37|    14|           42|
|   B|2026-01-03 08:45:11|    35|           77|
|   B|2026-01-03 09:10:59|    11|           88|
|   B|2026-01-04 18:32:45|    24|          112|
|   B|2026-01-05 10:18:20|    27|          139|
|   B|2026-01-05 10:55:33|    19|       

# 8

In [80]:
from pyspark.sql import SparkSession, functions as F 
from pyspark.sql.types import StructField, StructType, StringType, IntegerType


spark = SparkSession.builder.appName("test").getOrCreate()

schema = StructType([
    StructField("month", StringType()),
    StructField("product", StringType()),
    StructField("sales", IntegerType())
]
)

df = spark.read.csv("8.csv", header=True, schema=schema)

# pivots the table so that the product is part of the schema and sums up the sales per that product during a certain month
pivoted_table = df.groupBy("month").pivot("product").sum("sales").show()


+-----+---+---+---+---+---+
|month|  A|  B|  C|  D|  E|
+-----+---+---+---+---+---+
|  Oct| 30| 45| 55| 42| 70|
|  Sep| 28| 40| 50| 37| 65|
|  Dec| 40| 55| 65| 52| 80|
|  Aug| 25| 38| 45| 34| 60|
|  May| 20| 28| 33| 26| 45|
|  Jun| 18| 30| 38| 29| 50|
|  Feb| 12| 20| 25| 17| 28|
|  Nov| 35| 50| 60| 48| 75|
|  Mar| 14| 18| 30| 21| 35|
|  Jan| 10| 15| 22| 18| 30|
|  Apr| 16| 24| 27| 23| 40|
|  Jul| 22| 35| 42| 31| 55|
+-----+---+---+---+---+---+



# 9

In [81]:
from pyspark.sql import SparkSession, functions as F 
from pyspark.sql.types import StructField, StructType, StringType, IntegerType


spark = SparkSession.builder.appName("test").getOrCreate()



li = [("Alice", ["Math", "Science"]),("Bob", ["Jistory"])]
columns = ["name", "subjects"]

df = spark.createDataFrame(li, columns)

# use explode to git rid of the list of subjects and bring it down to a 3 tuple
df = df.withColumn("subject", F.explode("subjects")).drop("subjects")
df.show()
df.groupBy("name").agg(f.count("subject").alias("courses")).show()

+-----+-------+
| name|subject|
+-----+-------+
|Alice|   Math|
|Alice|Science|
|  Bob|Jistory|
+-----+-------+

+-----+-------+
| name|courses|
+-----+-------+
|Alice|      2|
|  Bob|      1|
+-----+-------+



# 10

In [82]:
from pyspark.sql import SparkSession, functions as F 
from pyspark.sql.types import StructField, StructType, StringType, IntegerType


spark = SparkSession.builder.appName("test").getOrCreate()

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType())
])

# read it with multiline, telling Pyspark to parse a single object or array that spans accross multiple lines
df = spark.read.option("multiline", True).json("10.json")

# explode the list of customers so that there is one entry per one customer
df = df.withColumn("customer", F.explode("customers")).drop("customers")

# select all the nested fields and give them new aliases
flat_df = df.select(F.col("customer.customer.id").alias("id"), 
                    F.col("customer.customer.name").alias("name"),
                    F.col("customer.customer.city").alias("city"),
                    F.col("customer.customer.state").alias("state"),
                    F.col("customer.customer.age").alias("age"),
                    F.col("customer.customer.orders").alias("orders"),
                    F.col("customer.customer.total_spent").alias("total_spent")
                )
flat_df.show()

+---+----------------+------------+-----+---+------+-----------+
| id|            name|        city|state|age|orders|total_spent|
+---+----------------+------------+-----+---+------+-----------+
|  1|      John Smith|          NY|   NY| 34|     5|     450.75|
|  2|   Sarah Johnson|     Chicago|   IL| 29|     8|      920.5|
|  3|   Michael Brown| Los Angeles|   CA| 41|    12|    1530.25|
|  4|     Emily Davis|     Houston|   TX| 26|     3|      210.0|
|  5|    David Wilson|     Phoenix|   AZ| 38|     7|      780.4|
|  6|Jessica Martinez|Philadelphia|   PA| 31|    10|    1250.75|
|  7| Robert Anderson| San Antonio|   TX| 45|    15|     2100.9|
|  8|   Amanda Thomas|   San Diego|   CA| 27|     4|      360.3|
|  9|  Daniel Jackson|      Dallas|   TX| 52|    20|    3200.15|
| 10|    Sophia White|     Seattle|   WA| 36|     9|     1100.6|
+---+----------------+------------+-----+---+------+-----------+

